# 🧟 Zombie Outbreak Spatial Spread Simulator
**Platform:** Microsoft Fabric (Runtime 1.3) · **Spatial engine:** ArcGIS GeoAnalytics for Microsoft Fabric

This notebook simulates the spread of a zombie outbreak from a user-defined origin across an H3 hexagon grid and writes the results as spatially-sorted GeoParquet to OneLake.

**Model in one paragraph:** the study area is tiled with H3 cells. Each cell gets a human population by areal interpolation from Esri's USA Census Tracts, plus two vulnerability indices built from CDC/ATSDR Social Vulnerability Index (SVI 2022) tract variables. Those indices scale how fast humans get bitten and how well they fight back. Every simulated day, (1) a fraction of each cell's zombies wander into neighbouring cells, weighted by distance; (2) fractional zombie arrivals are stochastically rounded, so the front advances realistically rather than diffusing infinitely; (3) within each cell, zombies convert humans and humans destroy zombies, in the style of the classic SZR model (Munz et al., 2009).

**Before you run:**
1. A tenant admin must enable *ArcGIS GeoAnalytics* in Fabric tenant settings.
2. Attach a Lakehouse (outputs go to its `Files/` area).
3. Use Runtime 1.3 and **disable the Native Execution Engine** for this notebook's environment; GeoAnalytics doesn't currently support NEE.
4. The next cell is tagged as the **parameters cell**, so a pipeline or `notebookutils.notebook.run()` can override any value.

In [ ]:
# ===================== PARAMETERS (tagged as parameter cell) =====================

# --- Outbreak origin ---
OUTBREAK_NAME      = "Patient Zero - Atlanta, GA"
OUTBREAK_LAT       = 33.7490          # WGS84 latitude
OUTBREAK_LON       = -84.3880         # WGS84 longitude
OUTBREAK_START     = "2026-10-31T00:00:00"   # ISO timestamp of first report
INITIAL_ZOMBIES    = 25.0

# --- Simulation extent & resolution ---
SIM_DAYS           = 30
H3_RESOLUTION      = 7        # 7 ≈ 5.2 km² cells; 8 ≈ 0.74 km² (≈7x more cells)
STUDY_RADIUS_KM    = 150      # grid built inside this geodesic radius of the origin
NEIGHBOR_RADIUS_KM = 6.0      # zombies can wander to cells whose centers are within this distance

# --- Epidemiology (per day) ---
INFECTION_RATE     = 0.90     # beta: humans bitten/converted per effective contact
ZOMBIE_KILL_RATE   = 0.35     # alpha: zombies destroyed by humans per effective contact
ZOMBIE_MOBILITY    = 0.10     # fraction of a cell's zombies that wander out each day
ZOMBIE_DECAY_RATE  = 0.01     # zombies that simply fall apart each day
STOCHASTICITY      = 0.30     # sd of lognormal noise on daily infections (0 = deterministic)
INFESTED_THRESHOLD = 1.0      # zombies needed for a cell to count as "infested"
OVERRUN_FRACTION   = 0.50     # zombie share of original population for "overrun"

# --- Population (US only when USE_REAL_POPULATION is True) ---
USE_REAL_POPULATION = True
TRACTS_URL = "https://services.arcgis.com/P3ePLMYs2RVChkJx/ArcGIS/rest/services/USA_Census_Tracts/FeatureServer/0"
TRACTS_SR  = 4326     # native spatial reference of the tracts layer (used for the extent filter)
SVI_URL    = "https://services3.arcgis.com/ZvidGQkLaDJxRSJ2/ArcGIS/rest/services/CDC_ATSDR_Social_Vulnerability_Index_2022_USA/FeatureServer/2"
SVI_SR     = 3857     # native spatial reference of the SVI tract layer
# Which population count to spread across cells:
#   "tract_2022"  Esri tracts POPULATION (2022)      "tract_2020"  Esri tracts POPULATION_2020
#   "svi_acs"     SVI E_TOTPOP (2018-22 ACS)          "svi_daytime" SVI E_DAYPOP (LandScan daytime: a work-hours outbreak)
POPULATION_SOURCE   = "tract_2022"
DEFAULT_DENSITY_KM2 = 60.0    # synthetic density when USE_REAL_POPULATION is False

# --- Vulnerability (built from SVI national percentiles, 0-1) ---
#   "custom"      two indices from the weighted variables below (recommended)
#   "svi_overall" use SVI RPL_THEMES for both indices (note: includes the minority-status theme)
#   "none"        ignore vulnerability
VULNERABILITY_MODE = "custom"
# Contact index -> raises the bite rate (people packed together)
CONTACT_WEIGHTS_JSON = '{"EPL_CROWD": 1.0, "EPL_MUNIT": 1.0, "EPL_GROUPQ": 1.0}'
# Escape/defence index -> raises the bite rate AND lowers the zombie kill rate (harder to flee or fight)
ESCAPE_WEIGHTS_JSON  = '{"EPL_AGE65": 1.0, "EPL_DISABL": 1.0, "EPL_NOVEH": 1.0, "EPL_AGE17": 0.5, "EPL_MOBILE": 0.5}'
# Other usable SVI percentiles: EPL_POV150, EPL_UNEMP, EPL_UNINSUR, EPL_LIMENG, EPL_SNGPNT, EPL_HBURD, EPL_NOHSDP
CONTACT_EFFECT = 0.40   # max +/- change to bite rate from the contact index (0.4 = +/-40%)
ESCAPE_EFFECT  = 0.40   # max +/- change to bite rate from the escape index
DEFENSE_EFFECT = 0.40   # max +/- change to zombie kill rate from the escape index

# --- Reproducibility & output ---
RANDOM_SEED          = 42
OUTPUT_ROOT          = "Files/zombie_outbreak"
RUN_ID               = ""     # blank = auto-generate
PARTITION_REGION_RES = 4      # coarse H3 level used to cluster rows spatially in files
TARGET_FILES_PER_DAY = 4

## 1. Setup and parameter validation

In [ ]:
import datetime as dt
import json
import uuid
from functools import reduce

from pyspark.sql import functions as F, Window
import geoanalytics_fabric
from geoanalytics_fabric.sql import functions as ST

geoanalytics_fabric.enable_warnings()

assert -90 <= OUTBREAK_LAT <= 90 and -180 <= OUTBREAK_LON <= 180, "Origin must be valid WGS84 lat/lon"
assert 0 <= H3_RESOLUTION <= 10, "Resolutions above 10 will create a very large grid"
assert 0 < ZOMBIE_MOBILITY < 1 and SIM_DAYS > 0
assert PARTITION_REGION_RES < H3_RESOLUTION
assert POPULATION_SOURCE in ("tract_2022", "tract_2020", "svi_acs", "svi_daytime")
assert VULNERABILITY_MODE in ("custom", "svi_overall", "none")

CONTACT_WEIGHTS = json.loads(CONTACT_WEIGHTS_JSON)
ESCAPE_WEIGHTS  = json.loads(ESCAPE_WEIGHTS_JSON)
for name, w in {**CONTACT_WEIGHTS, **ESCAPE_WEIGHTS}.items():
    assert name.startswith("EPL_") and w >= 0, f"Weights must be non-negative SVI EPL_ percentile fields: {name}"

RUN_ID = RUN_ID or dt.datetime.utcnow().strftime("%Y%m%dT%H%M%S") + "_" + uuid.uuid4().hex[:6]
START_TS = dt.datetime.fromisoformat(OUTBREAK_START)
OUT = f"{OUTPUT_ROOT}/run_id={RUN_ID}"

print(f"Run {RUN_ID}: {OUTBREAK_NAME} at ({OUTBREAK_LAT}, {OUTBREAK_LON}), {SIM_DAYS} days, H3 res {H3_RESOLUTION}")
print(f"Output: {OUT}")

## 2. Outbreak origin and H3 study grid

In [ ]:
origin_df = (spark.createDataFrame([(OUTBREAK_NAME, float(OUTBREAK_LON), float(OUTBREAK_LAT))],
                                   ["name", "lon", "lat"])
             .withColumn("geometry", ST.point("lon", "lat", sr=4326)))

origin_h3 = origin_df.select(ST.bin_id(ST.h3_bin("geometry", H3_RESOLUTION)).alias("h3_id")).first()["h3_id"]
origin_pt = origin_df.select("geometry")

# Geodesic circle around the origin, then every H3 cell that touches it
study_area = origin_df.select(ST.geodesic_buffer("geometry", STUDY_RADIUS_KM * 1000).alias("area"))

cells = (study_area
         .select(F.explode(ST.h3_bins("area", H3_RESOLUTION)).alias("bin"))
         .repartition(64)
         .select(ST.bin_id("bin").alias("h3_id"),
                 ST.bin_geometry("bin").alias("cell_geom"),
                 ST.bin_center("bin").alias("center"))
         .withColumn("area_km2", ST.geodesic_area("cell_geom") / 1e6)
         .withColumn("region_id", ST.bin_id(ST.h3_bin("center", PARTITION_REGION_RES))))

cells = cells.crossJoin(origin_pt.select(F.col("geometry").alias("origin"))) \
             .withColumn("dist_from_origin_km", ST.geodesic_distance("center", "origin") / 1000) \
             .drop("origin").cache()

print(f"{cells.count():,} H3 cells in study area; origin cell = {origin_h3}")

## 3. Population and vulnerability from census tracts
1. Read Esri USA Census Tracts and CDC SVI 2022 tracts, filtered to the study area's extent in each layer's native spatial reference.
2. Join them on the 11-digit tract `FIPS` and build the two vulnerability indices per tract (SVI uses -999 for missing values, treated as null).
3. **Areal interpolation:** intersect every H3 cell with the tracts it overlaps and give it each tract's population in proportion to the overlapping area. Cell indices are population-weighted averages of the overlapping tracts. Esri's tracts are clipped to the coastline, so water cells get zero people.

In [ ]:
def read_feature_service(url, extent_sr):
    # Read only features in the study-area extent (in the layer's own SR); fall back to a full read if the filter returns nothing
    ext = (study_area.select(ST.transform("area", extent_sr).alias("g"))
           .select(ST.min_x("g").alias("xmin"), ST.min_y("g").alias("ymin"),
                   ST.max_x("g").alias("xmax"), ST.max_y("g").alias("ymax")).first())
    extent = f"{ext.xmin}, {ext.ymin}, {ext.xmax}, {ext.ymax}"
    df = spark.read.format("feature-service").option("extent", extent).load(url)
    if df.limit(1).count() == 0:
        print(f"Extent filter returned no rows for {url}; reading the full layer instead")
        df = spark.read.format("feature-service").load(url)
    return df

def weighted_index(weights):
    # Weighted mean of SVI percentiles, skipping nulls; null if every input is missing
    num = reduce(lambda a, b: a + b, [F.coalesce(F.col(c) * w, F.lit(0.0)) for c, w in weights.items()])
    den = reduce(lambda a, b: a + b, [F.when(F.col(c).isNotNull(), F.lit(float(w))).otherwise(0.0) for c, w in weights.items()])
    return F.when(den > 0, num / den)

if USE_REAL_POPULATION:
    tracts = (read_feature_service(TRACTS_URL, TRACTS_SR)
              .select(F.trim("FIPS").alias("FIPS"), "STATE_ABBR", "POPULATION", "POPULATION_2020",
                      ST.transform("shape", 4326).alias("tract_geom"))
              .join(study_area, ST.intersects(F.col("tract_geom"), F.col("area"))).drop("area"))

    svi_fields = sorted(set(CONTACT_WEIGHTS) | set(ESCAPE_WEIGHTS) |
                        {"RPL_THEMES", "RPL_THEME1", "RPL_THEME2", "RPL_THEME4", "E_TOTPOP", "E_DAYPOP"})
    svi = read_feature_service(SVI_URL, SVI_SR).select(F.trim("FIPS").alias("FIPS"), "COUNTY", *svi_fields)
    for c in svi_fields:
        svi = svi.withColumn(c, F.when(F.col(c) >= 0, F.col(c).cast("double")))   # -999 -> null

    pop_expr = {"tract_2022": "POPULATION", "tract_2020": "POPULATION_2020",
                "svi_acs": "E_TOTPOP", "svi_daytime": "E_DAYPOP"}[POPULATION_SOURCE]

    tract_attrs = (tracts.join(F.broadcast(svi), "FIPS", "left")
                   .withColumn("tract_pop", F.coalesce(F.col(pop_expr).cast("double"), F.col("POPULATION").cast("double"), F.lit(0.0)))
                   .withColumn("tract_area_m2", ST.geodesic_area("tract_geom")))

    if VULNERABILITY_MODE == "custom":
        tract_attrs = (tract_attrs.withColumn("contact_idx", weighted_index(CONTACT_WEIGHTS))
                                  .withColumn("escape_idx",  weighted_index(ESCAPE_WEIGHTS)))
    elif VULNERABILITY_MODE == "svi_overall":
        tract_attrs = (tract_attrs.withColumn("contact_idx", F.col("RPL_THEMES"))
                                  .withColumn("escape_idx",  F.col("RPL_THEMES")))
    else:
        tract_attrs = (tract_attrs.withColumn("contact_idx", F.lit(None).cast("double"))
                                  .withColumn("escape_idx",  F.lit(None).cast("double")))
    tract_attrs = tract_attrs.cache()
    print(f"{tract_attrs.count():,} tracts in study area; "
          f"{tract_attrs.where('RPL_THEMES IS NULL').count():,} without an SVI match")

    # Areal interpolation: split each tract's population across the H3 cells it overlaps
    pieces = (cells.select("h3_id", "cell_geom")
              .join(tract_attrs, ST.intersects(F.col("cell_geom"), F.col("tract_geom")))
              .withColumn("piece_m2", ST.geodesic_area(ST.intersection("cell_geom", "tract_geom")))
              .withColumn("piece_pop", F.col("tract_pop") * F.col("piece_m2") / F.col("tract_area_m2")))

    def pop_weighted(col):
        return (F.sum(F.col("piece_pop") * F.col(col)) /
                F.sum(F.when(F.col(col).isNotNull(), F.col("piece_pop")))).alias(col)

    cell_vuln = (pieces.groupBy("h3_id").agg(
                    F.sum("piece_pop").alias("population"),
                    pop_weighted("contact_idx"), pop_weighted("escape_idx"),
                    pop_weighted("RPL_THEMES").alias("svi_overall"),
                    F.max_by("FIPS", "piece_pop").alias("main_tract_fips"),
                    F.max_by("COUNTY", "piece_pop").alias("county"),
                    F.max_by("STATE_ABBR", "piece_pop").alias("state")))

    cells_pop = (cells.join(cell_vuln, "h3_id", "left")
                 .fillna({"population": 0.0}))   # no tract overlap = water or outside the US
else:
    cells_pop = (cells.withColumn("population", F.lit(DEFAULT_DENSITY_KM2) * F.col("area_km2") * F.exp(F.randn(RANDOM_SEED) * 0.8))
                 .withColumn("contact_idx", F.lit(None).cast("double"))
                 .withColumn("escape_idx", F.lit(None).cast("double"))
                 .withColumn("svi_overall", F.lit(None).cast("double"))
                 .withColumn("main_tract_fips", F.lit(None).cast("string"))
                 .withColumn("county", F.lit(None).cast("string"))
                 .withColumn("state", F.lit(None).cast("string")))

# Turn indices (0-1 percentiles, 0.5 = national median) into rate multipliers centred on 1.0
c_dev = (F.coalesce(F.col("contact_idx"), F.lit(0.5)) - 0.5) * 2   # -1 .. +1
e_dev = (F.coalesce(F.col("escape_idx"),  F.lit(0.5)) - 0.5) * 2
cells_pop = (cells_pop
             .withColumn("population", F.round("population"))
             .withColumn("density_km2", F.col("population") / F.col("area_km2"))
             .withColumn("beta_mult",  F.greatest(F.lit(0.05), (1 + CONTACT_EFFECT * c_dev) * (1 + ESCAPE_EFFECT * e_dev)))
             .withColumn("alpha_mult", F.greatest(F.lit(0.05), 1 - DEFENSE_EFFECT * e_dev))
             .cache())

cells_pop.agg(F.sum("population").alias("total_population"),
              F.round(F.avg("contact_idx"), 3).alias("avg_contact_idx"),
              F.round(F.avg("escape_idx"), 3).alias("avg_escape_idx"),
              F.round(F.min("beta_mult"), 2).alias("min_bite_mult"), F.round(F.max("beta_mult"), 2).alias("max_bite_mult"),
              F.round(F.min("alpha_mult"), 2).alias("min_kill_mult"), F.round(F.max("alpha_mult"), 2).alias("max_kill_mult")).show()

## 4. Neighbor graph (who can wander where)
A geodesic buffer around each cell centre is spatially joined to other centres. Weights decay with distance and are normalised so a source cell's outflow sums to 1.

In [ ]:
centers = cells.select("h3_id", "center")
src = centers.select(F.col("h3_id").alias("src_id"), F.col("center").alias("src_center"),
                     ST.geodesic_buffer("center", NEIGHBOR_RADIUS_KM * 1000).alias("reach"))
dst = centers.select(F.col("h3_id").alias("dst_id"), F.col("center").alias("dst_center"))

decay_m = NEIGHBOR_RADIUS_KM * 1000 / 2.0
edges = (src.join(dst, ST.intersects(src.reach, dst.dst_center))
         .where("src_id != dst_id")
         .withColumn("dist_m", ST.geodesic_distance("src_center", "dst_center"))
         .withColumn("w", F.exp(-F.col("dist_m") / decay_m))
         .withColumn("w_norm", F.col("w") / F.sum("w").over(Window.partitionBy("src_id")))
         .select("src_id", "dst_id", "w_norm")
         .cache())

print(f"{edges.count():,} neighbour edges")

## 5. Run the simulation

In [ ]:
state = (cells_pop.select(
            "h3_id", "beta_mult", "alpha_mult",
            F.col("population").alias("N"),
            F.when(F.col("h3_id") == origin_h3, F.lit(INITIAL_ZOMBIES)).otherwise(F.lit(0.0)).alias("Z"))
         .withColumn("S", F.greatest(F.col("N") - F.col("Z"), F.lit(0.0)))
         .withColumn("R", F.lit(0.0))
         .select("h3_id", "beta_mult", "alpha_mult", "N", "S", "Z", "R")
         .localCheckpoint())

def snapshot(df, day):
    # Keep only cells the outbreak has touched, to keep output compact
    return df.where("Z > 0 OR R > 0").drop("beta_mult", "alpha_mult").withColumn("day", F.lit(day))

history = [snapshot(state, 0)]

for day in range(1, SIM_DAYS + 1):
    # 1) Movement: zombies wander to neighbours
    active = state.where("Z > 0").select(F.col("h3_id").alias("src_id"),
                                         (F.col("Z") * ZOMBIE_MOBILITY).alias("Z_out"))
    inflow = (edges.join(active, "src_id")
              .groupBy("dst_id").agg(F.sum(F.col("Z_out") * F.col("w_norm")).alias("Z_in"))
              .withColumnRenamed("dst_id", "h3_id"))

    s = (state.join(inflow, "h3_id", "left").fillna({"Z_in": 0.0})
         .withColumn("Z_m", F.col("Z") * (1 - ZOMBIE_MOBILITY) + F.col("Z_in"))
         # 2) Stochastic rounding: a 0.3 zombie becomes 1 zombie 30% of the time, else 0
         .withColumn("Z_m", F.when(F.col("Z_m") >= 1, F.col("Z_m"))
                             .when(F.rand(RANDOM_SEED + day) < F.col("Z_m"), F.lit(1.0))
                             .otherwise(F.lit(0.0)))
         # 3) Within-cell SZR interactions
         .withColumn("contact", F.col("S") * F.col("Z_m") / F.greatest(F.col("S") + F.col("Z_m"), F.lit(1.0)))
         .withColumn("noise", F.exp(F.randn(RANDOM_SEED * 1000 + day) * STOCHASTICITY - STOCHASTICITY ** 2 / 2))
         .withColumn("new_inf", F.least(F.col("S"), INFECTION_RATE * F.col("beta_mult") * F.col("contact") * F.col("noise")))
         .withColumn("destroyed", F.least(F.col("Z_m"),
                                          ZOMBIE_KILL_RATE * F.col("alpha_mult") * F.col("contact") + ZOMBIE_DECAY_RATE * F.col("Z_m"))))

    state = (s.select("h3_id", "beta_mult", "alpha_mult", "N",
                      (F.col("S") - F.col("new_inf")).alias("S"),
                      F.greatest(F.col("Z_m") + F.col("new_inf") - F.col("destroyed"), F.lit(0.0)).alias("Z"),
                      (F.col("R") + F.col("destroyed")).alias("R"))
             .localCheckpoint())   # truncate lineage each iteration

    history.append(snapshot(state, day))
    if day % 5 == 0 or day == SIM_DAYS:
        agg = state.agg(F.sum("Z").alias("Z"), F.sum(F.when(F.col("Z") >= INFESTED_THRESHOLD, 1).otherwise(0)).alias("cells")).first()
        print(f"Day {day:>3}: {agg['Z']:,.0f} zombies across {agg['cells']:,} infested cells")

## 6. Build output tables

In [ ]:
daily = reduce(lambda a, b: a.unionByName(b), history)

cell_attrs = cells_pop.select("h3_id", "cell_geom", "center", "region_id", "area_km2", "population",
                              "density_km2", "county", "state", "main_tract_fips", "dist_from_origin_km",
                              "contact_idx", "escape_idx", "svi_overall", "beta_mult", "alpha_mult")

cells_daily = (daily.join(cell_attrs, "h3_id")
    .withColumn("run_id", F.lit(RUN_ID))
    .withColumn("timestamp", F.expr(f"timestamp'{START_TS.isoformat(sep=' ')}' + make_interval(0,0,0,day)"))
    .withColumn("zombie_share", F.when(F.col("N") > 0, F.col("Z") / F.col("N")).otherwise(0.0))
    .withColumn("status", F.when(F.col("zombie_share") >= OVERRUN_FRACTION, "overrun")
                           .when(F.col("Z") >= INFESTED_THRESHOLD, "infested")
                           .when(F.col("R") > 0, "cleared")
                           .otherwise("trace")))

# Arrival "isochrones": first day each cell became infested
arrival = (cells_daily.where(F.col("Z") >= INFESTED_THRESHOLD)
           .groupBy("h3_id").agg(F.min("day").alias("first_infested_day"),
                                 F.max("Z").alias("peak_zombies"))
           .join(cell_attrs, "h3_id")
           .withColumn("run_id", F.lit(RUN_ID)))

# Daily outbreak front: footprint polygon, hull, mean centre, reach and speed
infested = cells_daily.where(F.col("Z") >= INFESTED_THRESHOLD)
front = (infested.groupBy("day", "timestamp")
         .agg(ST.aggr_union("cell_geom").alias("front_geom"),
              ST.aggr_convex_hull("center").alias("hull_geom"),
              ST.aggr_mean_center("center").alias("mean_center"),
              F.count("*").alias("infested_cells"),
              F.sum("Z").alias("total_zombies"),
              F.sum("R").alias("zombies_destroyed"),
              F.max("dist_from_origin_km").alias("front_reach_km"))
         .withColumn("spread_km_per_day",
                     F.col("front_reach_km") - F.lag("front_reach_km").over(Window.orderBy("day")))
         .withColumn("run_id", F.lit(RUN_ID)))

display(front.select("day", "infested_cells", "total_zombies", "front_reach_km", "spread_km_per_day").orderBy("day"))

## 7. Write spatially optimised GeoParquet
Rows are clustered by a coarse H3 region, then sorted by fine H3 id, so each Parquet row group covers a compact area. Explicit bbox columns give any reader (Spark, DuckDB, etc.) min/max statistics for spatial predicate pushdown.

In [ ]:
def add_bbox(df, geom_col):
    return (df.withColumn("bbox_xmin", ST.min_x(geom_col)).withColumn("bbox_ymin", ST.min_y(geom_col))
              .withColumn("bbox_xmax", ST.max_x(geom_col)).withColumn("bbox_ymax", ST.max_y(geom_col)))

def write_geoparquet(df, path, partition_cols=(), cluster_cols=(), num_files=None):
    if cluster_cols:
        n = num_files or max(1, df.rdd.getNumPartitions())
        df = df.repartitionByRange(n, *partition_cols, *cluster_cols) \
               .sortWithinPartitions(*partition_cols, *cluster_cols)
    w = df.write.format("geoparquet").mode("overwrite")
    if partition_cols:
        w = w.partitionBy(*partition_cols)
    w.save(path)
    print(f"Wrote {path}")

# GeoParquet tables should carry one geometry column; keep others as WKT if needed
cells_daily_out = add_bbox(cells_daily.withColumn("center_wkt", ST.as_text("center")).drop("center"), "cell_geom")
arrival_out     = add_bbox(arrival.withColumn("center_wkt", ST.as_text("center")).drop("center"), "cell_geom")
front_out       = add_bbox(front.withColumn("hull_wkt", ST.as_text("hull_geom"))
                                .withColumn("mean_center_wkt", ST.as_text("mean_center"))
                                .drop("hull_geom", "mean_center"), "front_geom")

write_geoparquet(cells_daily_out, f"{OUT}/cells_daily", partition_cols=("day",),
                 cluster_cols=("region_id", "h3_id"), num_files=TARGET_FILES_PER_DAY * (SIM_DAYS + 1))
write_geoparquet(arrival_out, f"{OUT}/arrival", cluster_cols=("region_id", "h3_id"), num_files=8)

# Static grid: population + vulnerability per cell, handy for mapping the inputs
grid_out = add_bbox(cell_attrs.withColumn("center_wkt", ST.as_text("center")).drop("center")
                    .withColumn("run_id", F.lit(RUN_ID)), "cell_geom")
write_geoparquet(grid_out, f"{OUT}/grid_population_vulnerability", cluster_cols=("region_id", "h3_id"), num_files=8)
write_geoparquet(front_out.coalesce(1), f"{OUT}/front_daily")
origin_df.withColumn("run_id", F.lit(RUN_ID)).write.format("geoparquet").mode("overwrite").save(f"{OUT}/origin")

## 8. Visual check and read-back test

In [ ]:
arr = spark.read.format("geoparquet").load(f"{OUT}/arrival")
arr.select("cell_geom", "first_infested_day").st.plot(
    cmap_values="first_infested_day", cmap="inferno_r", figsize=(8, 8), basemap="light")

In [ ]:
# Spatial read-back: bbox columns let Parquet skip row groups outside a ~20 km window
pad = 0.2
window = (spark.read.format("geoparquet").load(f"{OUT}/cells_daily")
          .where(F.col("day") == SIM_DAYS)
          .where((F.col("bbox_xmax") >= OUTBREAK_LON - pad) & (F.col("bbox_xmin") <= OUTBREAK_LON + pad) &
                 (F.col("bbox_ymax") >= OUTBREAK_LAT - pad) & (F.col("bbox_ymin") <= OUTBREAK_LAT + pad)))
print(f"Cells near origin on final day: {window.count():,}")
window.select("h3_id", "status", "S", "Z", "R").orderBy(F.desc("Z")).show(10)

## 9. Did vulnerability change the outcome?
Cells are split into quintiles of the escape/defence index (1 = least vulnerable). Compare how soon each group was reached and how much of its population was zombified by the final day.

In [ ]:
final = cells_daily.where(F.col("day") == SIM_DAYS).select("h3_id", F.col("Z").alias("Z_final"))
populated = cell_attrs.where("population > 0 AND escape_idx IS NOT NULL")

by_quintile = (populated
    .join(arrival.select("h3_id", "first_infested_day"), "h3_id", "left")
    .join(final, "h3_id", "left").fillna({"Z_final": 0.0})
    .withColumn("escape_quintile", F.ntile(5).over(Window.orderBy("escape_idx")))
    .groupBy("escape_quintile")
    .agg(F.count("*").alias("cells"),
         F.sum("population").alias("population"),
         F.round(F.avg("escape_idx"), 2).alias("avg_escape_idx"),
         F.round(F.avg("first_infested_day"), 1).alias("avg_first_infested_day"),
         F.round(F.sum("Z_final") / F.sum("population"), 4).alias("zombified_share_final"),
         F.round(F.avg(F.when(F.col("first_infested_day").isNotNull(), 1).otherwise(0)), 3).alias("share_of_cells_reached"))
    .orderBy("escape_quintile"))
display(by_quintile)

In [ ]:
grid = spark.read.format("geoparquet").load(f"{OUT}/grid_population_vulnerability")
grid.where("population > 0").select("cell_geom", "escape_idx").st.plot(
    cmap_values="escape_idx", cmap="YlOrRd", figsize=(8, 8), basemap="light")

## Running with different origins
Call from another notebook or pipeline, overriding parameters:
```python
notebookutils.notebook.run("Zombie_Outbreak_Spread", 3600,
    {"OUTBREAK_NAME": "Seattle", "OUTBREAK_LAT": 47.6062, "OUTBREAK_LON": -122.3321, "SIM_DAYS": 45})
```